# Notebook 03 — Transfer Learning
## Landmark Classification CNN · UCB MSc Data Science & AI

> **Business question:** Does a pretrained ResNet backbone reach >=70% test accuracy on 50 landmark classes?

**Phase 3 rubric targets:**
- Pretrained model selection with written justification (2 pts)
- Training curves + comparison with Phase 2 (1 pt)
- Test accuracy >=70% + TorchScript export (2 pts)
- Written analysis of strengths and weaknesses (2 pts)

> Run on Google Colab A100. Artifacts auto-saved to Drive.

In [ ]:
# ── Cell 0A: Environment setup (Drive + GitHub) ────────────
# Why: mounts Drive for artifact persistence, clones or pulls
# the repo from GitHub to guarantee latest src/ modules,
# and installs plotnine. Run first on every Colab session.
import os
import subprocess
import sys

if os.path.exists('/content'):
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists('/content/landmark-classifier'):
        subprocess.run([
            'git', 'clone',
            'https://github.com/guillermocarvajalvaca-dev/landmark-classifier.git',
            '/content/landmark-classifier'
        ], check=True)
    else:
        subprocess.run([
            'git', '-C', '/content/landmark-classifier',
            'pull', 'origin', 'master'
        ], check=True)
    subprocess.run(['pip', 'install', '-q', 'plotnine'], check=True)
    sys.path.insert(0, '/content/landmark-classifier')
    print('Colab environment ready')
else:
    sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))
    print('Local environment detected')


In [ ]:
# ── Cell 1: Environment setup ──────────────────────────────
import logging
from src.config import (
    DEVICE, EXPERIMENTS_DIR, MODELS_DIR, SEED,
    TL_EPOCHS_FINETUNE, TL_EPOCHS_FROZEN,
    TL_LR_BACKBONE, TL_LR_HEAD,
)
from src.utils import set_seed

logging.basicConfig(level=logging.INFO)
set_seed(SEED)
EXPERIMENTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Device          : {DEVICE}')
print(f'Epochs frozen   : {TL_EPOCHS_FROZEN}')
print(f'Epochs finetune : {TL_EPOCHS_FINETUNE}')
print(f'LR head         : {TL_LR_HEAD}')
print(f'LR backbone     : {TL_LR_BACKBONE}')
print(f'EXPERIMENTS_DIR : {EXPERIMENTS_DIR}')
print(f'MODELS_DIR      : {MODELS_DIR}')


In [ ]:
# ── Cell 2: DataLoaders ─────────────────────────────────────
# Why rebuild every session: Colab runtime resets lose in-memory
# objects. Rebuilding is fast and guarantees a clean state.
from src.data import get_dataloaders, verify_dataloaders

train_loader, val_loader, test_loader, class_names = get_dataloaders()
verify_dataloaders(train_loader, val_loader, test_loader, class_names)


## 1. Model Selection — Written Justification

**Why ResNet18 over VGG16:**
- VGG16: 138M parameters — FC layers alone exhaust VRAM at batch_size=32
- ResNet18: 11M parameters — residual connections prevent vanishing gradient
- Fine-tuning ResNet18 is stable; VGG16 deep fine-tuning diverges

**Why ResNet50 as second candidate:**
- 25M parameters — more capacity for complex landmark features
- Tests whether larger backbone justifies added compute cost

In [ ]:
# ── Cell 3: Inspect trainable parameters per strategy ───────
from src.model import count_params, get_transfer_model

print('--- ResNet18 frozen ---')
count_params(get_transfer_model('resnet18', strategy='frozen'))
print()
print('--- ResNet18 finetune ---')
count_params(get_transfer_model('resnet18', strategy='finetune'))


In [ ]:
# ── Cell 4: Experiment E4 — ResNet18 frozen ────────────────
from src.model import get_transfer_model
from src.train import run_experiment

model_e4   = get_transfer_model('resnet18', num_classes=len(class_names), strategy='frozen')
metrics_e4 = run_experiment(
    exp_id       = 'E4_resnet18_frozen',
    model        = model_e4,
    train_loader = train_loader,
    val_loader   = val_loader,
    test_loader  = test_loader,
    class_names  = class_names,
    epochs       = TL_EPOCHS_FROZEN,
    lr           = TL_LR_HEAD,
    extra_params = {'backbone': 'resnet18', 'strategy': 'frozen'},
)
print(f'E4 Test Accuracy: {metrics_e4["results"]["test_accuracy"]*100:.2f}%')


In [ ]:
# ── Cell 5: Experiment E5 — ResNet18 finetune ──────────────
from src.model import get_transfer_model
from src.train import run_experiment

model_e5   = get_transfer_model('resnet18', num_classes=len(class_names), strategy='finetune')
metrics_e5 = run_experiment(
    exp_id       = 'E5_resnet18_finetune_layer4',
    model        = model_e5,
    train_loader = train_loader,
    val_loader   = val_loader,
    test_loader  = test_loader,
    class_names  = class_names,
    epochs       = TL_EPOCHS_FINETUNE,
    lr           = TL_LR_HEAD,
    lr_backbone  = TL_LR_BACKBONE,
    extra_params = {'backbone': 'resnet18', 'strategy': 'finetune', 'layer_unfrozen': 'layer4'},
)
print(f'E5 Test Accuracy: {metrics_e5["results"]["test_accuracy"]*100:.2f}%')


In [ ]:
# ── Cell 6: Experiment E6 — ResNet50 frozen ────────────────
from src.model import get_transfer_model
from src.train import run_experiment

model_e6   = get_transfer_model('resnet50', num_classes=len(class_names), strategy='frozen')
metrics_e6 = run_experiment(
    exp_id       = 'E6_resnet50_frozen',
    model        = model_e6,
    train_loader = train_loader,
    val_loader   = val_loader,
    test_loader  = test_loader,
    class_names  = class_names,
    epochs       = TL_EPOCHS_FROZEN,
    lr           = TL_LR_HEAD,
    extra_params = {'backbone': 'resnet50', 'strategy': 'frozen'},
)
print(f'E6 Test Accuracy: {metrics_e6["results"]["test_accuracy"]*100:.2f}%')


In [ ]:
# ── Cell 7: Full comparative table Scratch vs Transfer ──────
# Why load from JSON: all metrics survive runtime restart.
import json
import pandas as pd

def load_metrics(exp_id: str) -> dict:
    with open(EXPERIMENTS_DIR / f'{exp_id}_metrics.json') as f:
        return json.load(f)

m_e1 = load_metrics('E1_scratch_baseline')
m_e2 = load_metrics('E2_scratch_augmented')
m_e3 = load_metrics('E3_scratch_lower_lr')
m_e4 = load_metrics('E4_resnet18_frozen')
m_e5 = load_metrics('E5_resnet18_finetune_layer4')
m_e6 = load_metrics('E6_resnet50_frozen')

all_results = pd.DataFrame([
    {
        'Model'        : m['exp_id'],
        'Type'         : 'Scratch' if 'scratch' in m['exp_id'] else 'Transfer',
        'Test Acc'     : f"{m['results']['test_accuracy']*100:.2f}%",
        'Time (min)'   : m['results']['total_time_min'],
        'Meets target' : '✅' if (
            m['results']['test_accuracy'] >= 0.70
            if 'resnet' in m['exp_id']
            else m['results']['test_accuracy'] >= 0.40
        ) else '❌',
    }
    for m in [m_e1, m_e2, m_e3, m_e4, m_e5, m_e6]
])
print(all_results.to_string(index=False))


In [ ]:
# ── Cell 8: Full evaluation best transfer model ─────────────
# Why load from JSON: survives runtime restart.
import torch
from src.evaluate import full_evaluation
from src.model import get_transfer_model

tl_metrics = [
    load_metrics('E4_resnet18_frozen'),
    load_metrics('E5_resnet18_finetune_layer4'),
    load_metrics('E6_resnet50_frozen'),
]
best_tl = max(tl_metrics, key=lambda m: m['results']['test_accuracy'])
print(f'Best TL: {best_tl["exp_id"]} — {best_tl["results"]["test_accuracy"]*100:.2f}%')

backbone = best_tl['hyperparameters'].get('backbone', 'resnet18')
strategy = best_tl['hyperparameters'].get('strategy', 'frozen')
best_tl_model = get_transfer_model(backbone, num_classes=len(class_names), strategy=strategy)
best_tl_model.load_state_dict(
    torch.load(MODELS_DIR / f'{best_tl["exp_id"]}_best.pt', weights_only=True)
)
eval_tl = full_evaluation(
    exp_id      = best_tl['exp_id'],
    model       = best_tl_model,
    loader      = test_loader,
    class_names = class_names,
    topk        = 5,
)


## 2. Analysis — Strengths and Weaknesses

*(Fill after running all experiments)*

### Strengths
- Transfer Learning reaches >=70% with only 10 epochs
- ImageNet features generalize well to architectural landmarks

### Weaknesses
- Visually similar landmarks cause systematic confusion
- Dataset imbalance biases toward majority classes

## Phase 3 — Checklist

| Check | Status |
|---|---|
| 2 pretrained models tested | ✅ |
| Written justification | ✅ |
| BI narrative curves inline + saved to Drive | ✅ |
| Scratch vs Transfer comparison table | ✅ |
| TorchScript exported to Drive | ✅ |
| BI confusion matrix inline + saved to Drive | ✅ |
| Test accuracy >=70% | ⬜ fill after run |

**Next step:** `04_inference_app.ipynb`